# Strava -> Bronze ingestion

Lands Strava API responses unchanged (raw JSON) in the multi-source `Bronze` lakehouse, schema `strava`.
Each source system owns one schema in Bronze (`strava`, later e.g. `garmin_connect`, `apple_health`); the `meta` schema
holds shared control tables (`source_registry`, `ingestion_run_log`). Table names are the source endpoint names.

| Bronze table (`strava.`) | Content | Load pattern |
|---|---|---|
| `athlete_activities` | `GET /athlete/activities` - activity summaries | Incremental (`after` watermark minus lookback) |
| `activities` | `GET /activities/{id}` - detailed activity (laps, splits, best efforts, calories, gear) | Not landed yet + last 2 days refreshed |
| `activities_streams` | `GET /activities/{id}/streams` - per-second time series | Not landed yet |
| `athlete` | `GET /athlete` - athlete profile (weight, FTP, bikes, shoes) | Snapshot each run |
| `athlete_zones` | `GET /athlete/zones` - heart-rate and power zones (needs `profile:read_all`) | Snapshot each run |
| `meta.ingestion_run_log` | One row per run and source with request usage, counts and remaining backfill | Append |

Strava read limits are 100 requests / 15 min and 1,000 / day. The client reads the rate-limit headers, waits for the next
15-minute window when needed, and stops cleanly when the daily or runtime budget is reached. Historical backfill of
details and streams therefore completes over several runs.

Strava OAuth credentials (`strava-client-id`, `strava-client-secret`, `strava-refresh-token`) are read from Azure Key Vault (`key_vault_url` parameter). Access tokens are kept in memory only; a rotated refresh token is written back to Key Vault.

In [ ]:
full_refresh = False
lookback_days = 7
max_detail_requests = 150
max_stream_requests = 150
refresh_recent_details_days = 2
max_runtime_minutes = 90
wait_on_rate_limit = True
key_vault_url = "https://your-key-vault.vault.azure.net/"

In [ ]:
from datetime import datetime, timezone, timedelta
from pyspark.sql import Row, functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, TimestampType, DateType
import hashlib
import json
import requests
import time
import uuid

spark.conf.set("spark.sql.session.timeZone", "UTC")

SOURCE_SYSTEM = "strava"
BRONZE_SCHEMA = SOURCE_SYSTEM                     # one Bronze schema per source system
META_SCHEMA = "meta"
API_BASE = "https://www.strava.com/api/v3"
# Bronze tables are named after the Strava API endpoint they come from.
ACTIVITIES_TABLE = "athlete_activities"          # GET /athlete/activities  (SummaryActivity)
ENTITY_TABLES = {
    "activity_detail": "activities",             # GET /activities/{id}         (DetailedActivity)
    "activity_streams": "activities_streams",    # GET /activities/{id}/streams (StreamSet)
    "athlete": "athlete",                        # GET /athlete                 (DetailedAthlete)
    "athlete_zones": "athlete_zones",            # GET /athlete/zones           (Zones)
}
RUN_LOG_TABLE = f"{META_SCHEMA}.ingestion_run_log"
STREAM_KEYS = "time,distance,heartrate,watts,cadence,velocity_smooth,altitude,grade_smooth,temp,moving"

BATCH_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)
DEADLINE = time.time() + max_runtime_minutes * 60

API_RAW_SCHEMA = StructType([
    StructField("source_system", StringType(), False),
    StructField("source_entity", StringType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("parent_record_id", StringType(), True),
    StructField("athlete_id", StringType(), True),
    StructField("api_endpoint", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("raw_payload", StringType(), True),
    StructField("payload_hash", StringType(), True),
    StructField("ingested_at", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
    StructField("batch_id", StringType(), False),
])

ACTIVITIES_SCHEMA = StructType([
    StructField("source_system", StringType(), False),
    StructField("source_entity", StringType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("athlete_id", StringType(), True),
    StructField("raw_payload", StringType(), True),
    StructField("api_page", LongType(), True),
    StructField("ingested_at", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
    StructField("batch_id", StringType(), False),
])

print(f"Strava bronze ingestion batch {BATCH_ID} (full_refresh={full_refresh})")

In [ ]:
class BudgetExhausted(Exception):
    pass


class StravaClient:
    """Small Strava API client that respects the 15-minute and daily read limits."""

    def __init__(self, access_token):
        self.session = requests.Session()
        self.session.headers.update({"Authorization": f"Bearer {access_token}"})
        self.requests_made = 0
        self.short_usage = 0
        self.short_limit = 100
        self.daily_usage = 0
        self.daily_limit = 1000

    def _update_limits(self, response):
        usage = response.headers.get("X-ReadRateLimit-Usage") or response.headers.get("X-RateLimit-Usage")
        limit = response.headers.get("X-ReadRateLimit-Limit") or response.headers.get("X-RateLimit-Limit")
        if usage and limit:
            self.short_usage, self.daily_usage = [int(v) for v in usage.split(",")[:2]]
            self.short_limit, self.daily_limit = [int(v) for v in limit.split(",")[:2]]

    def _wait_for_next_window(self):
        now = datetime.now(timezone.utc)
        next_window = now.replace(minute=(now.minute // 15) * 15, second=0, microsecond=0) + timedelta(minutes=15)
        wait_seconds = (next_window - now).total_seconds() + 5
        if not wait_on_rate_limit or time.time() + wait_seconds > DEADLINE:
            raise BudgetExhausted("15-minute Strava rate limit reached and waiting is not possible within the runtime budget")
        print(f"Rate limit window full ({self.short_usage}/{self.short_limit}); sleeping {int(wait_seconds)}s")
        time.sleep(wait_seconds)
        self.short_usage = 0

    def _ensure_budget(self):
        if time.time() > DEADLINE:
            raise BudgetExhausted("Runtime budget reached")
        if self.daily_usage >= self.daily_limit - 10:
            raise BudgetExhausted(f"Daily Strava read limit nearly reached ({self.daily_usage}/{self.daily_limit})")
        if self.short_usage >= self.short_limit - 3:
            self._wait_for_next_window()

    def get(self, path, params=None):
        response = None
        for attempt in range(3):
            self._ensure_budget()
            response = self.session.get(f"{API_BASE}{path}", params=params, timeout=90)
            self.requests_made += 1
            self._update_limits(response)
            if response.status_code == 429:
                self.short_usage = self.short_limit
                continue
            if response.status_code >= 500 and attempt < 2:
                time.sleep(10 * (attempt + 1))
                continue
            return response
        return response


def payload_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest() if text is not None else None


def fq(name):
    return f"{BRONZE_SCHEMA}.{name}"


def table_exists(name):
    return spark.catalog.tableExists(fq(name))


def read_table(name):
    return spark.table(fq(name))


def append_table(df, name, partition_col=None):
    writer = df.write.mode("append").format("delta").option("mergeSchema", "true")
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(fq(name))


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {META_SCHEMA}")

In [ ]:
# Strava credentials live only in Azure Key Vault - never in lakehouse tables, notebook outputs or source control.
# Secrets: strava-client-id, strava-client-secret, strava-refresh-token. The running identity needs
# "Key Vault Secrets Officer" on the vault (read + write, because Strava rotates refresh tokens).
def kv_get(name):
    try:
        return notebookutils.credentials.getSecret(key_vault_url, name)
    except Exception as exc:
        raise RuntimeError(
            f"Cannot read secret '{name}' from {key_vault_url}. Check that it exists (python tools/strava_auth.py) and that "
            "the identity running this notebook has the 'Key Vault Secrets Officer' role on the vault.") from exc


def kv_set(name, value):
    kv_token = notebookutils.credentials.getToken("keyvault")
    r = requests.put(
        f"{key_vault_url.rstrip('/')}/secrets/{name}?api-version=7.4",
        headers={"Authorization": f"Bearer {kv_token}"},
        json={"value": value, "contentType": "text/plain"},
        timeout=30,
    )
    if r.status_code >= 300:
        raise RuntimeError(f"Key Vault write failed for {name}: HTTP {r.status_code}")


response = requests.post(
    "https://www.strava.com/oauth/token",
    data={
        "client_id": kv_get("strava-client-id"),
        "client_secret": kv_get("strava-client-secret"),
        "grant_type": "refresh_token",
        "refresh_token": kv_get("strava-refresh-token"),
    },
    timeout=60,
)
if response.status_code != 200:
    raise RuntimeError(f"Strava token refresh failed: HTTP {response.status_code}. The Strava authorization was revoked or expired - re-authorize with: python tools/strava_auth.py")
token_payload = response.json()
if token_payload.get("refresh_token") and token_payload["refresh_token"] != kv_get("strava-refresh-token"):
    kv_set("strava-refresh-token", token_payload["refresh_token"])
    print("Rotated Strava refresh token stored in Key Vault")
access_token = token_payload["access_token"]
print("Strava access token obtained (kept in memory only)")

client = StravaClient(access_token)

In [ ]:
api_rows = []
counts = {"activities": 0, "athlete": 0, "athlete_zones": 0, "activity_detail": 0, "activity_streams": 0, "errors": 0}
stop_reason = None


def land(entity, record_id, parent_id, athlete_id, endpoint, response):
    text = response.text if response.status_code == 200 else (response.text or "")[:4000]
    now = datetime.now(timezone.utc)
    api_rows.append(Row(
        source_system=SOURCE_SYSTEM,
        source_entity=entity,
        source_record_id=record_id,
        parent_record_id=parent_id,
        athlete_id=athlete_id,
        api_endpoint=endpoint,
        http_status=int(response.status_code),
        raw_payload=text,
        payload_hash=payload_hash(text),
        ingested_at=now,
        ingestion_date=now.date(),
        batch_id=BATCH_ID,
    ))
    if response.status_code == 200:
        counts[entity] += 1
    else:
        counts["errors"] += 1
        print(f"{entity} {record_id}: HTTP {response.status_code} {(response.text or '')[:200]}")


def flush_api_rows(force=False):
    global api_rows
    if api_rows and (force or len(api_rows) >= 25):
        batch = spark.createDataFrame(api_rows, API_RAW_SCHEMA)
        for entity in {r["source_entity"] for r in api_rows}:
            append_table(batch.where(F.col("source_entity") == entity), ENTITY_TABLES[entity], "ingestion_date")
        api_rows = []


# 1) Activity summaries (incremental by start date)
after_epoch = None
if not full_refresh and table_exists(ACTIVITIES_TABLE):
    max_start = (read_table(ACTIVITIES_TABLE)
        .select(F.max(F.to_timestamp(F.get_json_object("raw_payload", "$.start_date"))).alias("m"))
        .collect()[0]["m"])
    if max_start is not None:
        after_epoch = int((max_start.replace(tzinfo=timezone.utc) - timedelta(days=lookback_days)).timestamp())

activity_rows = []
athlete_id = None
try:
    for page in range(1, 200):
        params = {"per_page": 200, "page": page}
        if after_epoch:
            params["after"] = after_epoch
        response = client.get("/athlete/activities", params)
        response.raise_for_status()
        records = response.json()
        if not records:
            break
        ingested_at = datetime.now(timezone.utc)
        for record in records:
            athlete = record.get("athlete") or {}
            record_athlete_id = str(athlete.get("id")) if athlete.get("id") is not None else None
            athlete_id = athlete_id or record_athlete_id
            activity_rows.append(Row(
                source_system=SOURCE_SYSTEM,
                source_entity="activities",
                source_record_id=str(record.get("id")),
                athlete_id=record_athlete_id,
                raw_payload=json.dumps(record, sort_keys=True),
                api_page=page,
                ingested_at=ingested_at,
                ingestion_date=ingested_at.date(),
                batch_id=BATCH_ID,
            ))
        if len(records) < 200:
            break
except BudgetExhausted as exc:
    stop_reason = str(exc)

if activity_rows:
    append_table(spark.createDataFrame(activity_rows, ACTIVITIES_SCHEMA), ACTIVITIES_TABLE, "ingestion_date")
counts["activities"] = len(activity_rows)
print(f"Activity summaries landed: {len(activity_rows)} (after={after_epoch})")

if athlete_id is None and table_exists(ACTIVITIES_TABLE):
    first = read_table(ACTIVITIES_TABLE).select("athlete_id").where("athlete_id is not null").limit(1).collect()
    athlete_id = first[0]["athlete_id"] if first else None

In [ ]:
# 2) Athlete profile and training zones (one call each per run; zones need the profile:read_all scope)
if stop_reason is None:
    try:
        response = client.get("/athlete")
        if response.status_code == 200 and response.json().get("id") is not None:
            athlete_id = str(response.json()["id"])
        land("athlete", athlete_id, None, athlete_id, "/athlete", response)
        response = client.get("/athlete/zones")
        land("athlete_zones", athlete_id, athlete_id, athlete_id, "/athlete/zones", response)
    except BudgetExhausted as exc:
        stop_reason = str(exc)
flush_api_rows(force=True)

In [ ]:
# 3) Detailed activities and streams for activities that are not landed yet (newest first)
summaries = (
    read_table(ACTIVITIES_TABLE)
    .select(
        "source_record_id",
        F.to_timestamp(F.get_json_object("raw_payload", "$.start_date")).alias("start_utc"),
        F.get_json_object("raw_payload", "$.manual").cast("boolean").alias("is_manual"),
        F.get_json_object("raw_payload", "$.moving_time").cast("long").alias("moving_time"),
    )
    .dropDuplicates(["source_record_id"])
)

def landed_ids(entity):
    table_name = ENTITY_TABLES[entity]
    if not table_exists(table_name):
        return spark.createDataFrame([], "source_record_id string")
    return read_table(table_name).where(F.col("http_status").isin(200, 404)).select("source_record_id").distinct()

recent_cutoff = F.current_timestamp() - F.expr(f"INTERVAL {int(refresh_recent_details_days)} DAYS")
detail_landed = landed_ids("activity_detail")
stream_landed = landed_ids("activity_streams")

detail_ids = [r["source_record_id"] for r in (
    summaries.join(detail_landed, "source_record_id", "left_anti")
    .unionByName(summaries.where(F.col("start_utc") >= recent_cutoff))
    .dropDuplicates(["source_record_id"])
    .orderBy(F.desc("start_utc"))
    .limit(int(max_detail_requests))
    .collect()
)]

stream_ids = [r["source_record_id"] for r in (
    summaries.where(~F.coalesce(F.col("is_manual"), F.lit(False)) & (F.col("moving_time") > 0))
    .join(stream_landed, "source_record_id", "left_anti")
    .orderBy(F.desc("start_utc"))
    .limit(int(max_stream_requests))
    .collect()
)]

print(f"Detail candidates: {len(detail_ids)}, stream candidates: {len(stream_ids)}")

work = [("activity_detail", i) for i in detail_ids] + [("activity_streams", i) for i in stream_ids]

if stop_reason is None:
    try:
        for entity, activity_id in work:
            if entity == "activity_detail":
                endpoint = f"/activities/{activity_id}"
                response = client.get(endpoint, {"include_all_efforts": "false"})
            else:
                endpoint = f"/activities/{activity_id}/streams"
                response = client.get(endpoint, {"keys": STREAM_KEYS, "key_by_type": "true", "series_type": "time"})
            land(entity, activity_id, activity_id, athlete_id, endpoint, response)
            flush_api_rows()
    except BudgetExhausted as exc:
        stop_reason = str(exc)
flush_api_rows(force=True)

In [ ]:
# Streams are only requested for recorded activities (manual entries and zero moving time have none), so count the same way.
remaining = {}
all_ids = summaries.select("source_record_id")
stream_ids_wanted = summaries.where(~F.coalesce(F.col("is_manual"), F.lit(False)) & (F.col("moving_time") > 0)).select("source_record_id")
for entity, wanted in [("activity_detail", all_ids), ("activity_streams", stream_ids_wanted)]:
    remaining[entity] = wanted.join(landed_ids(entity), "source_record_id", "left_anti").count()

result = {
    "status": "partial" if stop_reason else "success",
    "layer": "bronze",
    "batch_id": BATCH_ID,
    "counts": counts,
    "api_requests": client.requests_made,
    "daily_read_usage": f"{client.daily_usage}/{client.daily_limit}",
    "backfill_remaining": remaining,
    "stop_reason": stop_reason,
}

spark.createDataFrame([Row(
    batch_id=BATCH_ID,
    source_system=SOURCE_SYSTEM,
    started_at=RUN_STARTED_AT,
    finished_at=datetime.now(timezone.utc),
    status=result["status"],
    api_requests=int(client.requests_made),
    details_json=json.dumps(result),
)]).write.mode("append").format("delta").saveAsTable(RUN_LOG_TABLE)

for table_name in [ACTIVITIES_TABLE, *ENTITY_TABLES.values()]:
    if table_exists(table_name):
        spark.sql(f"OPTIMIZE {fq(table_name)}")

print(json.dumps(result, indent=2))
notebookutils.notebook.exit(json.dumps(result))